In [0]:
from pyspark.sql.functions import *



df = spark.table("gizmo_box.bronze.payments").withColumnRenamed("payment_date", "payment_timestamp").withColumnRenamed("payment_status", "payment_status_int")
df= df.withColumn("payments_date", date_format(col('payment_timestamp'), 'dd-MM-yyyy'))
df= df.withColumn("payments_time", date_format(col('payment_timestamp'), 'HH:mm:ss'))
df= df.drop("payment_timestamp")
df = df.withColumn("payment_status", when(col("payment_status_int") == 1, "Success") 
.when(col("payment_status_int") == 2, "Pending")
.when(col("payment_status_int") == 3, "Cancelled")
.when(col("payment_status_int") == 4, "Failed")).drop("payment_status_int")



display(df)



In [0]:
%sql

DROP table IF EXISTS gizmo_box.silver.payments;
CREATE Table IF NOT EXISTS gizmo_box.silver.payments AS
select
  payment_id,
  customer_id,
  payment_method,
  CAST(date_format(payment_date, 'HH:mm:ss') as date) as payment_time,
  date_format(payment_date, "yyyy-MM-dd") as payment_date,
  CASE payment_status
    when 1 THEN 'Success'
    when 2 THEN 'Pending'
    when 3 THEN 'Cancelled'
    when 4 THEN 'Failed'
  END as payment_status
from
  gizmo_box.bronze.payments